# NFL project: training and evaluation

<a href="https://colab.research.google.com/github/alcazarfran77-dev/NFL/blob/english/src/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

Do NFL Combine drills forecast how a prospect moves and performs in regular-season games, and does that say
anything about who becomes a starter? This notebook runs the whole project, from the raw data to the models
registered in MLflow:

1. sets up the environment (clones the repository in Colab and installs the pinned dependencies);
2. checks the raw data against the manifest and the data contract;
3. cleans the data and builds the movement and on-field performance metrics (`data_prep.py`);
4. trains and evaluates the models with nested cross-validation and registers them in MLflow (`train.py`);
5. shows the metrics, the statistical tests and the figures;
6. loads the registered models and uses them to predict.

The notebook calls exactly the same code as `make all`, so it gives the same numbers. The report with the
interpretation is in `CHANGELOG.md` and the reproduction guide in `README.md`. The raw tables can be Parquet,
CSV or compressed CSV: the pipeline detects each file's format and converts it before processing.

| Mode | What it does | Duration in Colab |
| --- | --- | --- |
| `smoke` | The whole pipeline with minimal cross-validation and grids; writes to its own folders (`data/smoke`, `reports/smoke`, `mlflow/smoke.db`) and leaves the release results alone | about 10 min |
| `full` | The release run (5 × 3 outer folds, full grids); rewrites `data/clean`, `reports/` and adds versions to the `mlflow/mlflow.db` registry | 60–90 min |
| `load_only` | Trains nothing: shows the release metrics and figures and loads the 17 registered models | 1–2 min |

## 1. Settings

`REPO_URL` and `BRANCH` already point at the project's English branch; just pick the mode. If the repository is
private, Colab needs credentials to clone it: use a URL with a personal access token
(`https://<token>@github.com/<user>/NFL.git`) or make the repository public. If you open the notebook from a
local copy of the repository (Jupyter or VS Code), the URL is not used: the notebook finds the project folder
by itself.

In [ ]:
REPO_URL = "https://github.com/alcazarfran77-dev/NFL.git"   # change it if you use your own copy of the repository
BRANCH = "english"                                  # "main" holds the Spanish v3.1.0
MODE = "smoke"                                      # "smoke", "full" or "load_only"

assert MODE in ("smoke", "full", "load_only"), "MODE must be 'smoke', 'full' or 'load_only'"

## 2. Environment

In Colab the repository is cloned into `/content/NFL` and the versions pinned in `requirements.txt` are
installed. If the installation changes libraries the runtime had already loaded in memory, the runtime restarts
once automatically: in that case run all the cells again (*Runtime → Run all*); the second time the
installation is skipped.

If Colab's Python version lacks one of the pinned versions, the latest compatible versions are installed. The
pipeline works the same, although the last decimals of some metrics could vary.

In [ ]:
import os, sys, subprocess, pathlib, logging

IN_COLAB = "google.colab" in sys.modules


def find_root(start: pathlib.Path):
    """Project folder: the first one, going up, that contains src/data_prep.py."""
    for c in [start, *start.parents]:
        if (c / "src" / "data_prep.py").exists():
            return c
    return None


ROOT = find_root(pathlib.Path.cwd())
if ROOT is None:
    target = pathlib.Path("/content/NFL") if IN_COLAB else pathlib.Path.cwd() / "NFL"
    if not target.exists():
        assert "<user>" not in REPO_URL, "Write your repository URL in REPO_URL (cell 1)"
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(target)], check=True)
    ROOT = target
os.chdir(ROOT)
print("Project folder:", ROOT)

flag = ROOT / ".dependencies_installed"
if IN_COLAB and not flag.exists():
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
    if r.returncode != 0:
        print("The pinned versions are not available for this Python version; "
              "installing the latest compatible ones.")
        packages = [l.split("==")[0] for l in open("requirements.txt", encoding="utf-8")
                    if l.strip() and not l.startswith("#")]
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
    flag.touch()
    loaded = [m for m in ("numpy", "pandas", "scipy", "sklearn", "matplotlib", "xgboost") if m in sys.modules]
    if loaded:
        print("Restarting the runtime to use the installed versions of", loaded,
              "-> run all the cells again.")
        os.kill(os.getpid(), 9)
print("Environment ready.")

In [ ]:
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
sys.path.insert(0, str(ROOT / "src"))
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)

import warnings
import pandas as pd
from IPython.display import Image, Markdown, display

import data_prep
import train

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

CONFIG = "configs/smoke.yaml" if MODE == "smoke" else "configs/config.yaml"
cfg = data_prep.load_config(CONFIG)
cfg_release = data_prep.load_config("configs/config.yaml")
data_prep.set_global_seed(cfg.seed)

# MLflow stores absolute paths: point them at this folder (does nothing if they already do).
print("MLflow store cells relocated:", train.relocate_store(cfg_release))
print(f"Pipeline version {data_prep.__version__} | mode {MODE} | configuration {CONFIG} | seed {cfg.seed}")

## 3. Raw data and data contract

The ten raw files are in `data/raw/`: the four small tables as Parquet and the six large ones as gzip-compressed
CSV, to respect GitHub's 100 MB per-file limit. Each file's format is detected from its first bytes, not from
its extension. The manifest stores the SHA-256 hash and the format of every file; if a file changed, the cell
says so and the data version (`data_version`) is no longer the report's.

The data contract (`configs/data_contract.yaml`) is the provider's spec in checkable form: types, ranges, keys,
foreign keys and coverage. Parquet columns are converted to the CSV encodings before the checks, so the same
contract applies to every format. Errors stop the pipeline; known deviations are warnings handled by the
business rules.

In [ ]:
import json

manifest = json.loads(cfg_release.path("manifest").read_text(encoding="utf-8"))
mismatched = data_prep.verify_manifest(cfg_release)
print("data_version:", manifest["data_version"],
      "| every file matches the manifest" if not mismatched else f"| do NOT match: {mismatched}")

files = pd.DataFrame([
    {"file": name, "stored as": e.get("stored_as", name), "format": e.get("format", "csv"), "records": e["rows"],
     "MB (content)": round(e["bytes"] / 1e6, 1), "MB on disk": round(e.get("stored_bytes", e["bytes"]) / 1e6, 1)}
    for name, e in manifest["files"].items()
])
display(files)

In [ ]:
summary = data_prep.stage_contract(cfg)
print(f"Contract: {summary['status']} — {summary['n_checks']} checks, "
      f"{summary['n_errors']} errors, {summary['n_warnings']} warnings")
print("Formats read:", summary.get("formats"))

report = pd.read_csv(cfg.path("metrics") / "data_contract_report.csv")
warnings_ = report[(report.severity == "warning") & (report.result == "fail")]
display(warnings_[["table", "column", "check", "detail", "handling"]].head(32))

## 4. Data cleaning and preparation (`data_prep.py`)

The data stage applies business rules R1–R16 (valid drill ranges, pick 260 for undrafted players, snap
anchoring, physical caps on the tracking, minimum plays per player, regular season, nullified plays, downs 1–4,
drill-tracking reliability) and builds:

- the prospects table with the `is_starter` target (started 50% or more of the available games);
- the per-play metrics from the game tracking and each player's Snap Burst Index;
- each player's on-field metrics over expected (`player_play`);
- the reliable features of the Combine drill tracking.

In `load_only` mode the clean data already stored in `data/clean/` is read.

In [ ]:
if MODE == "load_only":
    d = data_prep.load_processed(cfg)
else:
    d = data_prep.stage_data(cfg)   # about 2-3 minutes

audit = json.loads((cfg.path("metrics") / "data_audit.json").read_text(encoding="utf-8"))
keys = ["frames_in", "frames_out", "plays_kept", "players_tracked", "players_profiled", "ct_attempts_kept",
        "pp_rows_regular_season", "players_with_perf_index", "play_metric_anomalies", "contract_status"]
display(pd.Series({k: audit.get(k) for k in keys if k in audit}, name="value").to_frame())

p = d["prospects"]
print(f"{len(p)} prospects, {int(p.is_starter.sum())} starters ({p.is_starter.mean():.0%})")
display(p.groupby("combine_position", observed=True)
         .agg(prospects=("nfl_id", "size"), starters=("is_starter", "sum"),
              mean_forty=("forty", "mean"), drills_run=("n_drills_run", "mean"))
         .round(2))

In [ ]:
profiles = d["profiles"]
print(f"{len(profiles)} players with 3+ tracked plays; median plays per player:",
      int(profiles.n_plays.median()))
display(Markdown("**Share of prospects missing each drill, by position** (missing drills are imputed inside "
                 "each fold with the position median, and a flag marks the gap):"))
display(pd.read_csv(cfg.path("metrics") / "drill_missingness_by_position.csv", index_col=0).round(2))

## 5. Training and evaluation (`train.py`)

The `models` stage runs, in this order:

1. **Translation**: drill → game and drill → field correlations within position (with FDR correction),
   reliability of each metric (ICC), mediation on the defensive line and the starter GLMs.
2. **Regression**: Combine → SBI (all positions, defensive line, receivers) and Combine → on-field performance
   (four tasks). The position mean (baseline) competes against Ridge, Random Forest and XGBoost.
3. **Classification**: who becomes a starter? Logistic regression, Random Forest and XGBoost on nested feature
   sets (Combine; + draft; + movement; + on-field; + drill tracking).
4. **Statistical tests** on the same folds: corrected resampled t-test (Nadeau-Bengio), Wilcoxon with Holm,
   Friedman and DeLong.
5. **MLflow registration**: each run stores parameters, tags (`pipeline_version`, `data_version`,
   `git_commit`, seed), per-fold scores and out-of-fold predictions; champions and challengers are refitted on
   all rows and registered with aliases.

Cross-validation is nested: 5 folds × 3 repeats outside and 3 folds inside to choose hyperparameters (in
`smoke` mode: 2 × 1 and 2). Imputation and scaling are learned inside each fold.

In [ ]:
if MODE != "load_only":
    train.main(["--config", CONFIG, "--stage", "models"])   # smoke: about 6 min; full: 55-85 min
    train.main(["--config", CONFIG, "--stage", "figures"])
else:
    print("load_only mode: using the metrics and figures of the release run.")

## 6. Results

### 6.1 Regression: does the Combine forecast game movement and performance?

For each population: the baseline (position mean) and the best model by mean outer-fold RMSE, with the pooled
out-of-fold R² and its 95% bootstrap interval.

In [ ]:
M, F = cfg.path("metrics"), cfg.path("figures")

reg = pd.read_csv(M / "regression_summary.csv")
reg = reg[~reg.population.str.endswith("_ct")]
rows = []
for pop, g in reg.groupby("population", sort=False):
    base = g[g.model == "position_mean"].iloc[0]
    best = g[g.model != "position_mean"].sort_values("rmse_mean").iloc[0]
    rows.append({"population": pop, "target": base.target, "n": int(base.n), "best model": best.model,
                 "baseline RMSE": base.rmse_mean, "model RMSE": best.rmse_mean,
                 "change %": 100 * (best.rmse_mean / base.rmse_mean - 1),
                 "pooled R²": best.oof_r2, "95% CI": f"[{best.oof_r2_ci_low:.2f}, {best.oof_r2_ci_high:.2f}]"})
reg_table = pd.DataFrame(rows)
display(reg_table.round(3))

tests = pd.read_csv(M / "regression_pairwise_tests.csv")
vs_base = tests[(tests.metric == "rmse") & (tests.model_a == "position_mean")
                & ~tests.population.str.endswith("_ct")]
display(Markdown("**Each model against the baseline** (RMSE difference > 0 = the model has less error; "
                 "Holm-corrected p):"))
display(vs_base[["population", "model_b", "mean_diff_a_minus_b", "t_corrected", "p_corrected_t_holm",
                 "p_wilcoxon_holm"]].round(4))

In [ ]:
for f, title in [("fig10_regression_cv.png", "Per-fold error of each SBI model"),
                 ("fig18_onfield_regression.png", "Combine → on-field performance")]:
    if (F / f).exists():
        display(Markdown(f"**{title}**"), Image(filename=str(F / f), width=900))

### 6.2 Classification: who becomes a starter?

Mean outer-fold ROC-AUC by population, feature set and model. Draft capital is what moves the AUC most; game
movement and drill tracking barely change it.

In [ ]:
clf = pd.read_csv(M / "classification_summary.csv")
display(clf.pivot_table(index=["population", "feature_set"], columns="model", values="roc_auc_mean",
                        sort=False).round(3))

display(Markdown("**DeLong** (two AUCs on the same out-of-fold predictions):"))
display(pd.read_csv(M / "classification_delong.csv").round(4))

champion = clf[(clf.population == "all_prospects") & (clf.feature_set == "combine_draft")]
display(Markdown("**Pre-draft starter model (Combine + draft)**, metrics at threshold 0.5:"))
display(champion[["model", "roc_auc_mean", "roc_auc_std", "pr_auc_mean", "f1_mean", "recall_mean",
                  "precision_mean", "brier_mean"]].round(3))

In [ ]:
for f, title in [("fig11_classification_auc.png", "AUC by feature set"),
                 ("fig12_roc_curves.png", "Out-of-fold ROC curves"),
                 ("fig13_feature_importance.png", "Feature importance")]:
    if (F / f).exists():
        display(Markdown(f"**{title}**"), Image(filename=str(F / f), width=900))

### 6.3 Combine drill tracking: does it add anything?

The best model with and without the five reliable drill-tracking features, on the same players and folds.

In [ ]:
ct = pd.read_csv(M / "ct_best_vs_best.csv")
cols = ["task", "n", "model_without", "model_with", "r2_without", "r2_with", "r2_diff", "r2_p_corrected_t",
        "roc_auc_without", "roc_auc_with", "roc_auc_p_corrected_t", "p_delong"]
display(ct[[c for c in cols if c in ct.columns]].round(4))
if (F / "fig20_ct_value.png").exists():
    display(Image(filename=str(F / "fig20_ct_value.png"), width=900))

### 6.4 Translation gap on the defensive line

Observed in-game SBI minus the SBI the Combine profile predicts (out-of-fold). Positive = the player gets off
faster than he tested; negative = his profile says he can get off faster than he does, which points to
technique.

In [ ]:
gap = pd.read_csv(M / "translation_gap.csv")
dl = gap[(gap.population == "defensive_line") & (gap.n_plays >= 10)].sort_values("translation_gap")
cols = ["display_name", "nfl_position", "draft_year", "draft_round", "n_plays", "sbi", "predicted_sbi", "translation_gap"]
display(pd.concat([dl.tail(5)[::-1], dl.head(5)])[cols].round(1))
if (F / "fig14_translation_gap.png").exists():
    display(Image(filename=str(F / "fig14_translation_gap.png"), width=900))

### 6.5 All the figures

The report's 21 figures are written to the configuration's figures folder (`reports/figures/` in the release
run).

In [ ]:
figures = sorted(F.glob("fig*.png"))
print(len(figures), "figures in", F)
SHOW_ALL = False   # set to True to see them all here
for f in figures if SHOW_ALL else []:
    display(Markdown(f"**{f.name}**"), Image(filename=str(f), width=850))

## 7. Models registered in MLflow

Every champion or challenger model is registered with an alias (`champion` or `challenger`), its input
signature, an example and the `validation_status` tag. The models travel with the project code, so they load
with `mlflow.sklearn.load_model` without installing anything else.

In [ ]:
import mlflow
from mlflow import MlflowClient

mlflow.set_tracking_uri(cfg["mlflow"]["tracking_uri"])
client = MlflowClient()
registry = []
for rm in client.search_registered_models():
    for alias, version in rm.aliases.items():
        v = client.get_model_version(rm.name, version)
        run = client.get_run(v.run_id)
        registry.append({"model": rm.name, "alias": alias, "version": int(version),
                         "validation": v.tags.get("validation_status"),
                         "pipeline_version": run.data.tags.get("pipeline_version"),
                         "data_version": run.data.tags.get("data_version"),
                         "git_commit": run.data.tags.get("git_commit")})
registry = pd.DataFrame(registry).sort_values(["alias", "model"], ascending=[False, True])
print(len(registry), "models registered in", cfg["mlflow"]["tracking_uri"])
display(registry.reset_index(drop=True))

In [ ]:
uri = "models:/cgmt-starter-classifier@champion"
champion = mlflow.sklearn.load_model(uri)
columns = mlflow.models.get_model_info(uri).signature.inputs.input_names()
print("Pipeline steps:", [name for name, _ in champion.steps], "| input features:", len(columns))

# Illustration: probability of starting for some prospects of the 2025 class. These are in-sample predictions
# (the final model was trained on every prospect); the honest evaluation is the one in section 6.2.
prospects = d["prospects"]
example = prospects[prospects.draft_year == 2025].sort_values("draft_overall_pick").head(10).copy()
example["p_starter"] = champion.predict_proba(example[columns])[:, 1]
display(example[["display_name", "combine_position", "draft_overall_pick", "forty", "ten_yd_split",
                 "is_starter", "p_starter"]].round(3))

In [ ]:
# Defensive-line Snap Burst Index regressor: SBI projected from the Combine.
uri = "models:/cgmt-snap-burst-regressor-dl@champion"
regressor = mlflow.sklearn.load_model(uri)
columns = mlflow.models.get_model_info(uri).signature.inputs.input_names()
dl = prospects[prospects.combine_position == "DL"].copy()
dl["projected_sbi"] = regressor.predict(dl[columns])
display(dl.sort_values("projected_sbi", ascending=False)
          [["display_name", "draft_year", "combine_weight", "forty", "ten_yd_split", "projected_sbi"]]
          .head(10).round(2))

## 8. Next steps

- **Browse the runs:** in a local copy, `make ui` opens the MLflow interface with every run, per-fold metrics
  and the model registry.
- **New data:** copy the new files into `data/raw/` (with the spec's name or as parts `<name>_<n>`, in Parquet,
  CSV or compressed CSV) and run again in `full` mode; the contract validates them and the manifest records the
  new data version and each file's format. If a table is present in two formats, Parquet is used.
- **Interpretation:** the conclusions for scouts, general managers and coaches are in section 10 of
  `CHANGELOG.md`.